# 🧑‍⚖️ Golden Beta — LLM Judges

Deux LLM judges pour analyser les résultats de la config optimale sur le golden_beta.

## Judge 1 — Error Categorization
Pour chaque run avec un score bas (faithfulness < seuil ou pas de réponse satisfaisante) :
- Inspecte : query, contexte pré-selector, selector reasoning, contexte post-selector, réponse
- Output : catégorie d'erreur + explication

## Judge 2 — Beta Comparison  
Pour chaque run :
- Compare : query, réponse beta réelle, feedback utilisateur, nouvelle réponse optimale
- Output : problème résolu?, qualité (meilleure/égale/inférieure)

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os, sys, json, time
from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

DSN = os.getenv("TUNNEL_DSN") or os.getenv("SCALINGO_POSTGRESQL_URL") or os.getenv("PG_DSN")
conn = psycopg.connect(DSN, row_factory=dict_row)
cur = conn.cursor()
print("Connected ✅")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Config names to analyze
OPTIMAL_CONFIG = "v3_optim"                # ← The new optimal config
BETA_CONFIGS = ["v2_prod", "v3_prod"]      # ← Beta configs to compare against

# Judge model
JUDGE_MODEL = "gpt-4.1"  # Strong model for nuanced error categorization & comparison
JUDGE_TEMPERATURE = 0.0

# Thresholds for Judge 1 (which runs to analyze for errors)
FAITHFULNESS_THRESHOLD = 0.5  # Below this → investigate
ANALYZE_ALL = True            # True = analyze ALL runs (not just low-scoring)

# Rate limiting
DELAY_BETWEEN_CALLS = 0.3
BATCH_SIZE = 20

print(f"🧑‍⚖️ Judge model: {JUDGE_MODEL}")
print(f"📋 Optimal config: {OPTIMAL_CONFIG}")
print(f"📋 Beta configs: {BETA_CONFIGS}")

In [ ]:
# =============================================================================
# LLM CLIENT
# =============================================================================
from openai import OpenAI

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def call_judge(system_prompt: str, user_prompt: str, model: str = JUDGE_MODEL) -> str:
    """Call the LLM judge and return the response text."""
    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=JUDGE_TEMPERATURE,
                response_format={"type": "json_object"},
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                raise e

# Test
test = call_judge("Return JSON with key 'status'.", "Say hello.")
print(f"✅ Judge OK: {test[:60]}")

## 1. Charger les données

In [ ]:
# =============================================================================
# LOAD OPTIMAL RUNS + BETA RUNS + FEEDBACK
# =============================================================================

# 1. Load golden_beta questions with all metadata
cur.execute("""
    SELECT
        gq.id AS question_id,
        gq.question,
        gq.gold_answer,
        gq.gold_sources,
        gq.theme,
        gq.goldset_name,
        gq.original_turn_id
    FROM goldset_questions_v2 gq
    WHERE gq.tags @> ARRAY['golden_beta']
    ORDER BY gq.id
""")
questions = {r['question_id']: r for r in cur.fetchall()}
question_ids = list(questions.keys())
print(f"🏅 Golden Beta questions: {len(question_ids)}")

# 2. Load optimal runs
placeholders = ','.join(['%s'] * len(question_ids))
cur.execute(f"""
    SELECT
        gr.id AS run_id,
        gr.question_id,
        gr.config_name,
        gr.response,
        gr.retrieved_context,
        gr.config_params,
        gr.metrics,
        gr.retrieval_time_ms,
        gr.generation_time_ms
    FROM goldset_runs gr
    WHERE gr.question_id IN ({placeholders})
      AND gr.config_name = %s
    ORDER BY gr.question_id
""", question_ids + [OPTIMAL_CONFIG])
optimal_runs = {r['question_id']: r for r in cur.fetchall()}
print(f"📊 Optimal runs ({OPTIMAL_CONFIG}): {len(optimal_runs)}")

# 3. Load beta runs (for comparison)
beta_runs = {}  # {question_id: {config_name: run_data}}
for cfg in BETA_CONFIGS:
    cur.execute(f"""
        SELECT
            gr.id AS run_id,
            gr.question_id,
            gr.config_name,
            gr.response,
            gr.retrieved_context,
            gr.metrics
        FROM goldset_runs gr
        WHERE gr.question_id IN ({placeholders})
          AND gr.config_name = %s
        ORDER BY gr.question_id
    """, question_ids + [cfg])
    for r in cur.fetchall():
        qid = r['question_id']
        if qid not in beta_runs:
            beta_runs[qid] = {}
        beta_runs[qid][r['config_name']] = r

n_with_beta = sum(1 for qid in question_ids if qid in beta_runs)
print(f"📊 Questions with beta run: {n_with_beta}")

# 4. Load user feedback (from chat_feedbacks via original_turn_id)
feedbacks = {}
turn_ids = [q['original_turn_id'] for q in questions.values() if q.get('original_turn_id')]
if turn_ids:
    turn_placeholders = ','.join(['%s'] * len(turn_ids))
    cur.execute(f"""
        SELECT
            cf.turn_id,
            cf.helpful,
            cf.stars,
            cf.comment,
            cf.reasons_positive,
            cf.reasons_negative,
            cf.error_category,
            cf.ai_reason,
            cf.beta_scope
        FROM chat_feedbacks cf
        WHERE cf.turn_id::text IN ({turn_placeholders})
    """, turn_ids)
    for r in cur.fetchall():
        feedbacks[str(r['turn_id'])] = r

n_with_feedback = sum(
    1 for q in questions.values()
    if q.get('original_turn_id') and str(q['original_turn_id']) in feedbacks
)
print(f"📊 Questions with user feedback: {n_with_feedback}")

## 2. Judge 1 — Error Categorization

Pour chaque run (ou les runs mal notés), le judge analyse :
- La query
- Le contexte PRÉ-selector (toutes les sections reçues)
- Le reasoning du Selector
- Le contexte POST-selector (ce qui a été gardé)
- La réponse générée

Et catégorise l'erreur.

In [ ]:
# =============================================================================
# JUDGE 1 — ERROR CATEGORIZATION PROMPT
# =============================================================================

JUDGE1_SYSTEM = """Tu es un expert en évaluation de systèmes RAG (Retrieval-Augmented Generation) pour un assistant RH de la fonction publique française.

Ta mission : analyser une interaction RAG et déterminer si la réponse est satisfaisante ou non.
Si elle ne l'est pas, tu dois catégoriser l'erreur avec précision.

## Catégories d'erreur possibles

- **retrieval_miss** : La bonne information n'est pas dans le contexte récupéré. Le système n'a pas trouvé les bons documents.
- **selector_error** : La bonne information était dans le contexte pré-sélection, mais le LLM Selector l'a éliminée à tort.
- **generator_hallucination** : Le générateur a inventé des informations qui ne sont pas dans le contexte.
- **generator_incomplete** : Le générateur a omis des éléments importants présents dans le contexte.
- **generator_misinterpretation** : Le générateur a mal interprété le contexte ou la question.
- **question_ambiguous** : La question est ambiguë ou mal formulée, rendant la réponse difficile.
- **context_insufficient** : Le contexte est partiellement pertinent mais incomplet pour répondre pleinement.
- **correct** : La réponse est satisfaisante et correcte par rapport au contexte.

## Format de sortie (JSON)

```json
{
  "category": "<une des catégories ci-dessus>",
  "confidence": 0.0-1.0,
  "explanation": "Explication courte (2-3 phrases)",
  "is_satisfactory": true/false,
  "severity": "low|medium|high|critical",
  "improvement_suggestion": "Suggestion d'amélioration (optionnel)"
}
```

## Règles d'évaluation

1. Si la réponse est globalement correcte et utile → "correct"
2. Si l'info manque totalement du contexte → "retrieval_miss"
3. Si l'info était dans le contexte pré-selector mais pas post-selector → "selector_error"
4. Si le contexte est bon mais la réponse est fausse → "generator_hallucination" ou "generator_misinterpretation"
5. En cas de doute entre retrieval_miss et context_insufficient, préfère context_insufficient si le contexte est partiellement pertinent.
"""

MAX_CHARS_PER_ITEM = 8000  # GPT-4.1 handles 1M tokens, send full context

def build_judge1_prompt(question: str, response: str, context_items: list, gold_answer: str = None) -> str:
    """Build the user prompt for Judge 1 with FULL context (not truncated)."""
    ctx_text = ""
    if context_items:
        for i, item in enumerate(context_items):
            if isinstance(item, str):
                item = json.loads(item) if item.startswith('{') else {"text": item}
            source = item.get('source', item.get('doc_publisher', '?'))
            title = item.get('title', item.get('doc_title', '?'))
            score = item.get('score', '?')
            text = item.get('text', item.get('content', ''))
            if len(text) > MAX_CHARS_PER_ITEM:
                text = text[:MAX_CHARS_PER_ITEM] + f"\n[... tronqué, {len(text)} chars au total]"
            ctx_text += f"\n[{i+1}] Source: {source} | Titre: {title} | Score: {score}\n{text}\n"
    else:
        ctx_text = "(aucun contexte récupéré)"
    
    total_ctx_chars = sum(
        len(item.get('text', item.get('content', '')) if isinstance(item, dict) else '')
        for item in (context_items or [])
    )
    
    prompt = f"""## Question posée
{question}

## Contexte récupéré par le RAG (post-sélection) — {len(context_items or [])} items, {total_ctx_chars} chars total
{ctx_text}

## Réponse générée
{response}
"""
    
    if gold_answer:
        prompt += f"\n## Réponse de référence (gold_answer)\n{gold_answer}\n"
    else:
        prompt += "\n## Réponse de référence\n(non disponible — évalue uniquement la qualité intrinsèque et le grounding dans le contexte ci-dessus)\n"
    
    return prompt

print("✅ Judge 1 prompt ready")

In [ ]:
# =============================================================================
# RUN JUDGE 1 ON ALL OPTIMAL RUNS
# =============================================================================

judge1_results = []
errors = 0
total = len(optimal_runs)

print(f"🧑‍⚖️ Running Judge 1 on {total} runs...")

for i, (qid, run) in enumerate(optimal_runs.items()):
    q = questions[qid]
    
    # Parse context
    ctx = run.get('retrieved_context')
    if isinstance(ctx, str):
        try:
            ctx = json.loads(ctx)
        except:
            ctx = []
    if ctx is None:
        ctx = []
    
    # Build prompt
    prompt = build_judge1_prompt(
        question=q['question'],
        response=run.get('response', ''),
        context_items=ctx,
        gold_answer=q.get('gold_answer'),
    )
    
    try:
        raw = call_judge(JUDGE1_SYSTEM, prompt)
        result = json.loads(raw)
        result['question_id'] = qid
        result['run_id'] = run['run_id']
        judge1_results.append(result)
        
        if (i + 1) % BATCH_SIZE == 0:
            print(f"  [{i+1}/{total}] Last: {result['category']} (conf={result.get('confidence', '?')})")
    except Exception as e:
        errors += 1
        judge1_results.append({
            'question_id': qid,
            'run_id': run['run_id'],
            'category': 'error',
            'explanation': str(e),
            'is_satisfactory': None,
        })
    
    time.sleep(DELAY_BETWEEN_CALLS)

print(f"\n✅ Judge 1 complete: {len(judge1_results)} results, {errors} errors")

In [ ]:
# =============================================================================
# JUDGE 1 — ANALYSIS
# =============================================================================

df_j1 = pd.DataFrame(judge1_results)

print("=" * 60)
print("JUDGE 1 — ERROR CATEGORIZATION RESULTS")
print("=" * 60)

# Overall satisfaction
n_satisfactory = df_j1['is_satisfactory'].sum()
n_total = len(df_j1)
pct_satisfactory = n_satisfactory / n_total * 100 if n_total > 0 else 0
print(f"\n✅ Satisfactory: {n_satisfactory}/{n_total} ({pct_satisfactory:.1f}%)")
print(f"❌ Issues: {n_total - n_satisfactory}")

# Category distribution
print(f"\n📊 Error categories:")
cat_counts = df_j1['category'].value_counts()
for cat, cnt in cat_counts.items():
    pct = cnt / n_total * 100
    print(f"  {cat:30s} {cnt:4d} ({pct:5.1f}%)")

# Severity distribution (for non-correct)
if 'severity' in df_j1.columns:
    non_correct = df_j1[df_j1['category'] != 'correct']
    if len(non_correct) > 0:
        print(f"\n⚠️ Severity distribution (errors only):")
        sev_counts = non_correct['severity'].value_counts()
        for sev, cnt in sev_counts.items():
            print(f"  {sev:10s} {cnt:4d}")

# Show worst cases
print(f"\n🔍 Examples of errors:")
for cat in cat_counts.index:
    if cat == 'correct':
        continue
    examples = df_j1[df_j1['category'] == cat].head(3)
    print(f"\n  [{cat}]")
    for _, ex in examples.iterrows():
        qid = ex['question_id']
        q = questions.get(qid, {})
        print(f"    Q{qid}: {q.get('question', '?')[:70]}...")
        print(f"    → {ex.get('explanation', '?')[:100]}")

## 3. Judge 2 — Beta Comparison

Pour chaque question ayant un run beta ET un run optimal :
- Compare les deux réponses
- Tient compte du feedback utilisateur
- Évalue si le problème est résolu et si la qualité s'est améliorée

In [ ]:
# =============================================================================
# JUDGE 2 — BETA COMPARISON PROMPT
# =============================================================================

JUDGE2_SYSTEM = """Tu es un expert en évaluation de systèmes RAG (Retrieval-Augmented Generation) pour un assistant RH de la fonction publique française.

Ta mission : comparer deux réponses à la même question RH.
- **Réponse A** : la réponse du beta-test (version précédente du système)
- **Réponse B** : la nouvelle réponse optimisée (version améliorée du système)

Tu as aussi accès au feedback utilisateur sur la Réponse A (si disponible).

## Critères d'évaluation

1. **Exactitude** : la réponse est-elle factuelle et correcte ?
2. **Complétude** : la réponse couvre-t-elle tous les aspects de la question ?
3. **Clarté** : la réponse est-elle bien structurée et compréhensible ?
4. **Référencement** : les sources sont-elles citées correctement ?

## Format de sortie (JSON)

```json
{
  "quality_comparison": "better|equal|worse",
  "problem_resolved": true/false/null,
  "confidence": 0.0-1.0,
  "score_beta": 1-5,
  "score_optimal": 1-5,
  "explanation": "Explication de la comparaison (2-3 phrases)",
  "key_improvements": ["liste des améliorations notables"],
  "regressions": ["liste des régressions, si applicable"]
}
```

## Règles

- "problem_resolved" : true si le feedback utilisateur indiquait un problème ET la Réponse B le corrige, false si le problème persiste, null si pas de feedback.
- "quality_comparison" : "better" si B est globalement meilleure, "equal" si comparable, "worse" si B est en régression.
- Scores 1-5 : 1=très mauvais, 2=insuffisant, 3=passable, 4=bon, 5=excellent.
- Si la Réponse A est vide/absente, évalue uniquement la Réponse B.
"""

def build_judge2_prompt(question: str, beta_response: str, optimal_response: str,
                         feedback: dict = None) -> str:
    """Build the user prompt for Judge 2."""
    
    prompt = f"""## Question posée
{question}

## Réponse A (beta-test)
{beta_response or '(réponse non disponible)'}

## Réponse B (nouvelle version optimisée)
{optimal_response or '(réponse non disponible)'}
"""
    
    if feedback:
        fb_text = ""
        if feedback.get('stars'):
            fb_text += f"Note: {feedback['stars']}/5\n"
        if feedback.get('comment'):
            fb_text += f"Commentaire: {feedback['comment']}\n"
        if feedback.get('reasons_negative'):
            fb_text += f"Points négatifs: {feedback['reasons_negative']}\n"
        if feedback.get('reasons_positive'):
            fb_text += f"Points positifs: {feedback['reasons_positive']}\n"
        if feedback.get('error_category'):
            fb_text += f"Catégorie d'erreur identifiée: {feedback['error_category']}\n"
        if feedback.get('ai_reason'):
            fb_text += f"Analyse AI: {feedback['ai_reason']}\n"
        
        if fb_text:
            prompt += f"\n## Feedback utilisateur sur la Réponse A\n{fb_text}\n"
        else:
            prompt += "\n## Feedback utilisateur\n(aucun feedback)\n"
    else:
        prompt += "\n## Feedback utilisateur\n(aucun feedback disponible)\n"
    
    return prompt

print("✅ Judge 2 prompt ready")

In [ ]:
# =============================================================================
# RUN JUDGE 2 ON QUESTIONS WITH BOTH BETA AND OPTIMAL RUNS
# =============================================================================

judge2_results = []
errors = 0

# Build comparison pairs
comparison_pairs = []
for qid in question_ids:
    if qid not in optimal_runs:
        continue
    
    # Get beta response (prefer v3_prod, then v2_prod)
    beta_run = None
    if qid in beta_runs:
        for cfg in BETA_CONFIGS:
            if cfg in beta_runs[qid]:
                beta_run = beta_runs[qid][cfg]
                break
    
    # Get feedback
    q = questions[qid]
    fb = None
    if q.get('original_turn_id'):
        fb = feedbacks.get(str(q['original_turn_id']))
    
    comparison_pairs.append({
        'question_id': qid,
        'question': q['question'],
        'optimal_response': optimal_runs[qid].get('response', ''),
        'beta_response': beta_run.get('response', '') if beta_run else None,
        'beta_config': beta_run.get('config_name') if beta_run else None,
        'feedback': fb,
    })

total = len(comparison_pairs)
print(f"🧑‍⚖️ Running Judge 2 on {total} comparison pairs...")
print(f"  With beta response: {sum(1 for p in comparison_pairs if p['beta_response'])}")
print(f"  With feedback: {sum(1 for p in comparison_pairs if p['feedback'])}")

for i, pair in enumerate(comparison_pairs):
    prompt = build_judge2_prompt(
        question=pair['question'],
        beta_response=pair['beta_response'],
        optimal_response=pair['optimal_response'],
        feedback=pair['feedback'],
    )
    
    try:
        raw = call_judge(JUDGE2_SYSTEM, prompt)
        result = json.loads(raw)
        result['question_id'] = pair['question_id']
        result['beta_config'] = pair['beta_config']
        result['has_feedback'] = pair['feedback'] is not None
        judge2_results.append(result)
        
        if (i + 1) % BATCH_SIZE == 0:
            print(f"  [{i+1}/{total}] Last: {result['quality_comparison']} (beta={result.get('score_beta', '?')}, opt={result.get('score_optimal', '?')})")
    except Exception as e:
        errors += 1
        judge2_results.append({
            'question_id': pair['question_id'],
            'quality_comparison': 'error',
            'explanation': str(e),
        })
    
    time.sleep(DELAY_BETWEEN_CALLS)

print(f"\n✅ Judge 2 complete: {len(judge2_results)} results, {errors} errors")

In [ ]:
# =============================================================================
# JUDGE 2 — ANALYSIS
# =============================================================================

df_j2 = pd.DataFrame(judge2_results)

print("=" * 60)
print("JUDGE 2 — BETA COMPARISON RESULTS")
print("=" * 60)

# Quality comparison
print(f"\n📊 Quality comparison (optimal vs beta):")
comp_counts = df_j2['quality_comparison'].value_counts()
for comp, cnt in comp_counts.items():
    pct = cnt / len(df_j2) * 100
    emoji = {'better': '🟢', 'equal': '🟡', 'worse': '🔴', 'error': '⚫'}.get(comp, '❓')
    print(f"  {emoji} {comp:10s} {cnt:4d} ({pct:5.1f}%)")

# Problem resolution
if 'problem_resolved' in df_j2.columns:
    resolved = df_j2[df_j2['problem_resolved'].notna()]
    if len(resolved) > 0:
        n_resolved = resolved['problem_resolved'].sum()
        print(f"\n🔧 Problem resolution (questions with feedback):")
        print(f"  Resolved: {n_resolved}/{len(resolved)} ({n_resolved/len(resolved)*100:.1f}%)")
        print(f"  Not resolved: {len(resolved) - n_resolved}")

# Score comparison
if 'score_beta' in df_j2.columns and 'score_optimal' in df_j2.columns:
    valid = df_j2[(df_j2['score_beta'].notna()) & (df_j2['score_optimal'].notna())]
    if len(valid) > 0:
        avg_beta = valid['score_beta'].mean()
        avg_optimal = valid['score_optimal'].mean()
        print(f"\n⭐ Average scores:")
        print(f"  Beta:    {avg_beta:.2f}/5")
        print(f"  Optimal: {avg_optimal:.2f}/5")
        print(f"  Delta:   {avg_optimal - avg_beta:+.2f}")

# Show regressions
regressions = df_j2[df_j2['quality_comparison'] == 'worse']
if len(regressions) > 0:
    print(f"\n🔴 Regressions ({len(regressions)} questions):")
    for _, reg in regressions.head(5).iterrows():
        qid = reg['question_id']
        q = questions.get(qid, {})
        print(f"  Q{qid}: {q.get('question', '?')[:70]}...")
        print(f"    → {reg.get('explanation', '?')[:100]}")
        if reg.get('regressions'):
            print(f"    Régressions: {reg['regressions']}")

## 4. Synthèse pour le comité

In [ ]:
# =============================================================================
# COMBINED SYNTHESIS
# =============================================================================

print("=" * 70)
print("🏅 SYNTHÈSE — GOLDEN BETA EVALUATION")
print("=" * 70)

# Judge 1 summary
n_satisfactory = df_j1['is_satisfactory'].sum() if 'is_satisfactory' in df_j1.columns else 0
n_total_j1 = len(df_j1)
print(f"\n📋 Judge 1 — Qualité des réponses (config: {OPTIMAL_CONFIG})")
print(f"  Réponses satisfaisantes: {n_satisfactory}/{n_total_j1} ({n_satisfactory/n_total_j1*100:.1f}%)")
print(f"  Erreurs restantes:")
for cat, cnt in df_j1['category'].value_counts().items():
    if cat != 'correct':
        print(f"    {cat}: {cnt} ({cnt/n_total_j1*100:.1f}%)")

# Judge 2 summary
n_total_j2 = len(df_j2)
n_better = (df_j2['quality_comparison'] == 'better').sum() if len(df_j2) > 0 else 0
n_equal = (df_j2['quality_comparison'] == 'equal').sum() if len(df_j2) > 0 else 0
n_worse = (df_j2['quality_comparison'] == 'worse').sum() if len(df_j2) > 0 else 0

print(f"\n📋 Judge 2 — Comparaison avec le beta-test")
print(f"  🟢 Meilleure:  {n_better}/{n_total_j2} ({n_better/n_total_j2*100:.1f}%)")
print(f"  🟡 Égale:      {n_equal}/{n_total_j2} ({n_equal/n_total_j2*100:.1f}%)")
print(f"  🔴 Pire:       {n_worse}/{n_total_j2} ({n_worse/n_total_j2*100:.1f}%)")

# Cross-analysis: Judge 1 errors × Judge 2 comparison
print(f"\n📊 Analyse croisée:")
merged = df_j1.merge(df_j2, on='question_id', how='inner', suffixes=('_j1', '_j2'))
if len(merged) > 0:
    for cat in df_j1['category'].unique():
        if cat == 'correct':
            continue
        subset = merged[merged['category'] == cat]
        if len(subset) == 0:
            continue
        n_better_in_cat = (subset['quality_comparison'] == 'better').sum()
        n_worse_in_cat = (subset['quality_comparison'] == 'worse').sum()
        print(f"  {cat}: {len(subset)} questions → {n_better_in_cat} améliorées, {n_worse_in_cat} dégradées")

In [ ]:
# =============================================================================
# SAVE RESULTS
# =============================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Save Judge 1 results
j1_path = f"golden_beta_judge1_{timestamp}.csv"
df_j1_export = df_j1.copy()
df_j1_export['question'] = df_j1_export['question_id'].map(lambda qid: questions.get(qid, {}).get('question', ''))
df_j1_export.to_csv(j1_path, index=False)
print(f"💾 Judge 1 results saved to {j1_path}")

# Save Judge 2 results  
j2_path = f"golden_beta_judge2_{timestamp}.csv"
df_j2_export = df_j2.copy()
df_j2_export['question'] = df_j2_export['question_id'].map(lambda qid: questions.get(qid, {}).get('question', ''))
df_j2_export.to_csv(j2_path, index=False)
print(f"💾 Judge 2 results saved to {j2_path}")

# Save combined synthesis as JSON
synthesis = {
    "timestamp": timestamp,
    "optimal_config": OPTIMAL_CONFIG,
    "judge_model": JUDGE_MODEL,
    "n_questions": len(question_ids),
    "judge1": {
        "n_evaluated": n_total_j1,
        "n_satisfactory": int(n_satisfactory),
        "pct_satisfactory": round(n_satisfactory / n_total_j1 * 100, 1) if n_total_j1 > 0 else 0,
        "error_distribution": df_j1['category'].value_counts().to_dict(),
    },
    "judge2": {
        "n_compared": n_total_j2,
        "n_better": int(n_better),
        "n_equal": int(n_equal),
        "n_worse": int(n_worse),
        "pct_better": round(n_better / n_total_j2 * 100, 1) if n_total_j2 > 0 else 0,
    },
}

synth_path = f"golden_beta_synthesis_{timestamp}.json"
with open(synth_path, 'w') as f:
    json.dump(synthesis, f, indent=2, ensure_ascii=False)
print(f"💾 Synthesis saved to {synth_path}")

# Optionally save to DB
try:
    cur.execute("""
        INSERT INTO pipeline_eval_experiments
            (name, description, configs, aggregate, n_questions, theme)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, [
        f"Golden Beta Judges {timestamp}",
        f"Judge 1 (error categorization) + Judge 2 (beta comparison) on {OPTIMAL_CONFIG}",
        json.dumps({"optimal": OPTIMAL_CONFIG, "beta": BETA_CONFIGS, "judge_model": JUDGE_MODEL}),
        json.dumps(synthesis),
        len(question_ids),
        "LLM_Judge",
    ])
    conn.commit()
    print("💾 Synthesis saved to pipeline_eval_experiments")
except Exception as e:
    print(f"⚠️ Could not save to DB: {e}")

print("\n✅ All results saved!")

In [ ]:
# Close connection
cur.close()
conn.close()
print("Connection closed.")